# 05 - Hyperparameter tuning (validation set only)

Tunes Logistic Regression, KNN, SVM, Decision Tree, XGBoost and LightGBM on all **103 signal features**.

**Rules followed**
- Every setting is **fitted on the training set** and **scored on the validation set**.
- Best setting = highest validation **F1 of the abnormal class (1)**; validation ROC-AUC breaks ties.
- The **test set is not opened** here (see notebook 06).
- Imputer and scaler are inside the pipeline, so they are fitted on training data only.
- Random Forest reuses the setting already tuned with grouped cross-validation.

In [1]:
import itertools
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
TARGET_COLUMN = "figo_label"
FEATURE_PREFIXES = ("F_", "U_", "FU_")

# Project root = parent of the notebooks folder
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__)

# Load the TRAINING and VALIDATION files only. The test file is NOT opened in this notebook.
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")

# The 103 signal features (IDs and target are excluded)
feature_cols = []
for col in train_df.columns:
    if col.startswith(FEATURE_PREFIXES):
        feature_cols.append(col)
if len(feature_cols) != 103:
    raise ValueError("Expected 103 features, found " + str(len(feature_cols)))

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = train_df[TARGET_COLUMN].astype(int)
X_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_val = val_df[TARGET_COLUMN].astype(int)
print("Train:", X_train.shape, " Validation:", X_val.shape)

Python 3.13.12 | scikit-learn 1.9.1
Train: (14540, 103)  Validation: (3115, 103)


## Pipeline builder

In [2]:
def make_pipeline(name, params):
    """Imputer (+ StandardScaler for LR, KNN, SVM) + model. Everything is fitted on TRAIN only."""
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if name in ("Logistic Regression", "KNN", "SVM"):
        steps.append(("scaler", StandardScaler()))

    if name == "Logistic Regression":
        model = LogisticRegression(max_iter=3000, random_state=SEED, **params)
    elif name == "KNN":
        model = KNeighborsClassifier(**params)
    elif name == "SVM":
        model = SVC(kernel="rbf", random_state=SEED, **params)
    elif name == "Decision Tree":
        model = DecisionTreeClassifier(random_state=SEED, **params)
    elif name == "XGBoost":
        model = XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss", **params)
    else:
        model = LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, **params)

    steps.append(("model", model))
    return Pipeline(steps)


def grid(options):
    """Turn a dict of lists into a list of every combination (one dict per combination)."""
    keys = list(options.keys())
    combos = []
    for values in itertools.product(*[options[k] for k in keys]):
        combos.append(dict(zip(keys, values)))
    return combos

## Search spaces

In [ ]:
# Grid search for the small models, randomised search (20 settings) for XGBoost and LightGBM
search_space = {
    "Logistic Regression": grid({"C": [0.01, 0.1, 1, 10, 100]}),
    "KNN": grid({"n_neighbors": [3, 5, 7, 11, 15, 25], "weights": ["uniform", "distance"], "p": [1, 2]}),
    "SVM": grid({"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.001, 0.01]}),
    "Decision Tree": grid({"max_depth": [None, 5, 10, 15, 20], "min_samples_leaf": [1, 5, 10, 20],
                           "criterion": ["gini", "entropy"]}),
    "XGBoost": list(ParameterSampler({
        "n_estimators": [300, 600], "max_depth": [3, 5, 7], "learning_rate": [0.03, 0.1],
        "subsample": [0.8, 1.0], "colsample_bytree": [0.6, 0.8, 1.0], "min_child_weight": [1, 5]},
        n_iter=20, random_state=SEED)),
    "LightGBM": list(ParameterSampler({
        "n_estimators": [300, 600], "num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.1],
        "min_child_samples": [10, 20, 40], "subsample": [0.8, 1.0], "colsample_bytree": [0.6, 0.8, 1.0]},
        n_iter=20, random_state=SEED)),
}
for model_name in search_space:
    print(model_name, "->", len(search_space[model_name]), "settings")

Logistic Regression -> 5 settings
KNN -> 24 settings
SVM -> 12 settings
Decision Tree -> 40 settings
XGBoost -> 20 settings
LightGBM -> 20 settings


: 

## Run the search (fit on train, score on validation)

In [ ]:
rows = []
best = {}

for model_name in search_space:
    candidates = search_space[model_name]
    best_key = None

    for params in candidates:
        start = time.time()

        # Fit on the TRAINING set, score on the VALIDATION set
        pipe = make_pipeline(model_name, params)
        pipe.fit(X_train, y_train)
        val_pred = pipe.predict(X_val)
        if model_name == "SVM":
            val_score = pipe.decision_function(X_val)
        else:
            val_score = pipe.predict_proba(X_val)[:, 1]

        val_f1 = float(f1_score(y_val, val_pred, pos_label=1))
        val_auc = float(roc_auc_score(y_val, val_score))
        rows.append({"model": model_name, "params": json.dumps(params), "val_f1": val_f1,
                     "val_auc": val_auc, "seconds": round(time.time() - start, 1)})

        # Best = highest validation F1; ROC-AUC breaks ties
        key = (val_f1, val_auc)
        if best_key is None or key > best_key:
            best_key = key
            best[model_name] = {"params": params, "val_f1": val_f1, "val_auc": val_auc}

    print(model_name, "| best validation F1 %.4f, AUC %.4f" % (best[model_name]["val_f1"], best[model_name]["val_auc"]))
    print("   ", best[model_name]["params"])

Logistic Regression | best validation F1 0.9799, AUC 0.9982
    {'C': 10}
KNN | best validation F1 0.8533, AUC 0.9439
    {'n_neighbors': 11, 'weights': 'distance', 'p': 2}


## Random Forest (saved earlier result)

In [ ]:
# Random Forest was tuned earlier with RandomizedSearchCV (5-fold StratifiedGroupKFold on the
# training set). We reuse that saved setting instead of searching again.
rf_saved = json.loads((RESULTS_DIR / "random_forest_tuned_validation.json").read_text())
best["Random Forest"] = {
    "params": rf_saved["best_parameters"],
    "val_f1": rf_saved["validation_metrics"]["abnormal_f1"],
    "val_auc": None,
}
print("Random Forest:", best["Random Forest"]["params"])
print("Random Forest validation F1:", round(best["Random Forest"]["val_f1"], 4))

## Summary

In [ ]:
# Best setting per model, with the default-like validation F1 for comparison
summary_rows = []
for model_name in best:
    summary_rows.append({"model": model_name, "best_val_f1": best[model_name]["val_f1"],
                         "best_val_auc": best[model_name]["val_auc"], "best_params": json.dumps(best[model_name]["params"])})
summary_df = pd.DataFrame(summary_rows).sort_values("best_val_f1", ascending=False)
display(summary_df)

## Save results

In [ ]:
# Save the tuning results (validation only) for the final evaluation notebook
pd.DataFrame(rows).to_csv(RESULTS_DIR / "tuning_results.csv", index=False)
(RESULTS_DIR / "best_params.json").write_text(json.dumps(best, indent=1))
print("Saved:", RESULTS_DIR / "tuning_results.csv")
print("Saved:", RESULTS_DIR / "best_params.json")